# Titanic EDA, Data Preprocessing and model fitting
### Analystlab Africa Machine learning internship 
This notebook is for the week 4 of the analystlab africa machine learning internship.
in this notebook i will be performing exploratory data analysis, data preprocessing and model fitting and prediction on  Ames house price dataset.

### **About the Ames house price dataset**
Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this playground competition's dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence.

With 79 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this competition challenges you to predict the final price of each home.

Acknowledgments

The Ames Housing dataset was compiled by Dean De Cock for use in data science education. It's an incredible alternative for data scientists looking for a modernized and expanded version of the often cited Boston Housing dataset. 

### Goal
Build a model that predicts the price of a house using the features given.

we are given a dedicated train csv and test csv so we train on the train.csv and test on the test.csv.
check out the dataset [here](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/overview)

### Task Breakdown
- Step 1: Import Libraries

Import required Python libraries such as:
pandas
numpy
matplotlib / seaborn
sklearn

- Step 2: Load Dataset

Download the dataset
Load it into Jupyter Notebook using pandas


- Step 3: Explore the Dataset

Check:
first 5 rows,
missing values,
 data types,
column names

- Step 4: Select Features and Target

Example:
Features (X): rooms, size, bathrooms
Target (y): price

- Step 5: Split the Dataset

Split data into:
Training set
Testing set

Recommended:
80% training
20% testing


## Phase 1: Importing the needed libraries and dataset 

In [39]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_log_error
from sklearn.ensemble import GradientBoostingRegressor

In [20]:
#importing the dataset
direct = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/"
train_path = direct + "train.csv"
test_path = direct + "test.csv"
submission_path = direct + "sample_submission.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
submision_df = pd.read_csv(submission_path)

In [21]:
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Phase 2: Exploratry Data Analysis 

In [ ]:
#checking the shape of the dataset 
shape = train_df.shape 

print(f" There are {shape[0]} rows and {shape[1]} columns/features in the ames data \n thats a lot ")

In [ ]:
# checking the statistical distribution of the data 
display(train_df.describe())

In [ ]:
#checking for duplicate column 
print(f" The duplicated rows are: {train_df.duplicated().sum()}")

In [ ]:
# checking out the missing data in the dataset 
is_null_here = train_df.isna().sum() / len(train_df) * 100
is_nulldf = pd.DataFrame({"column": is_null_here.index, "Percent_missing":is_null_here.values}).sort_values(by ="Percent_missing" ,ascending = False)
display(is_nulldf.head(19))

- you can be sure that those columns with more than 50% of the rows missing will be removed cause theres no base to fill from 

## Phase 3: Data Preprocessing 

In [22]:

def clean_df(df, is_train=True, train_medians=None):
    """
    Cleans and encodes the dataset. 
    Returns the cleaned dataframe (and the median values if is_train=True).
    """
    # Create a copy to avoid SettingWithCopy warnings on the original data
    data = df.copy()

    # 1. Drop Unnecessary Columns
    cols_to_drop = ["Utilities", "LandSlope", "Condition1", "Condition2", 
                    "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd", 
                    "PoolQC", "Fence"]
    # Only drop columns if they actually exist in the dataframe
    data.drop(columns=[c for c in cols_to_drop if c in data.columns], inplace=True)

    # 2. Centralized Dictionary Mapping
    # This single dictionary replaces the massive chain of .replace() calls, 
    # making the code incredibly fast and easy to maintain.
    encoding_maps = {
        'MSZoning': {'A':0, 'C (all)':1, 'FV':2, 'I':3, 'RH':4, 'RL':5, 'RP':6, 'RM':7},
        'Street': {'Pave':0, 'Grvl':1},
        'Alley': {'Pave':1, 'Grvl':2},
        'LotShape': {'Reg':0, 'IR1':1, 'IR2':2, 'IR3':3},
        'LandContour': {'Lvl':0, 'Bnk':1, 'HLS':2, 'Low':3},
        'LotConfig': {'Inside':0, 'Corner':1, 'CulDSac':2, 'FR2':3, 'FR3':4},
        'Neighborhood': {'Blmngtn':0, 'Blueste':1, 'BrDale':2, 'BrkSide':3, 'ClearCr':4, 'CollgCr':5, 'Crawfor':6, 'Edwards':7, 'Gilbert':8, 'IDOTRR':9, 'MeadowV':10, 'Mitchel':11, 'NAmes':12, 'NoRidge':13, 'NPkVill':14, 'NridgHt':15, 'NWAmes':16, 'OldTown':17, 'SWISU':18, 'Sawyer':19, 'SawyerW':20, 'Somerst':21, 'StoneBr':22, 'Timber':23, 'Veenker':24},
        'BldgType': {'1Fam':0, '2fmCon':1, 'Duplex':2, 'TwnhsE':3, 'Twnhs':4},
        'HouseStyle': {'1Story':0, '2Story':1, '1.5Fin':2, '1.5Unf':3, '2.5Fin':4, '2.5Unf':5, 'SFoyer':6, 'SLvl':7},
        'MasVnrType': {'BrkCmn':0, 'BrkFace':1, 'CBlock':2, 'None':3, 'Stone':4},
        'ExterQual': {'Ex':0, 'Gd':1, 'TA':2, 'Fa':3, 'Po':4},
        'ExterCond': {'Ex':0, 'Gd':1, 'TA':2, 'Fa':3, 'Po':4},
        'Foundation': {'BrkTil':0, 'CBlock':1, 'Slab':2, 'Stone':3, 'Wood':4, 'PConc':5},
        'BsmtQual': {'Ex':0, 'Gd':1, 'TA':2, 'Fa':3, 'Po':4, 'Na':5},
        'BsmtCond': {'Ex':0, 'Gd':1, 'TA':2, 'Fa':3, 'Po':4, 'Na':5},
        'BsmtExposure': {'Gd':0, 'Av':1, 'Mn':2, 'No':3, 'Na':4},
        'BsmtFinType1': {'GLQ':0, 'ALQ':1, 'BLQ':2, 'Rec':3, 'LwQ':4, 'Unf':5, 'NA':6},
        'BsmtFinType2': {'GLQ':0, 'ALQ':1, 'BLQ':2, 'Rec':3, 'LwQ':4, 'Unf':5, 'NA':6},
        'Heating': {'Floor':0, 'GasA':1, 'GasW':2, 'Grav':3, 'OthW':4, 'Wall':5},
        'HeatingQC': {'Ex':0, 'Gd':1, 'TA':2, 'Fa':3, 'Po':4},
        'CentralAir': {'N':0, 'Y':1},
        'Electrical': {'SBrkr':0, 'FuseA':1, 'FuseF':2, 'FuseP':3, 'Mix':4},
        'KitchenQual': {'Ex':0, 'Gd':1, 'TA':2, 'Fa':3, 'Po':4},
        'Functional': {'Typ':0, 'Min1':1, 'Min2':2, 'Mod':3, 'Maj1':4, 'Maj2':5, 'Sev':6, 'Sal':7},
        'FireplaceQu': {'Ex':1, 'Gd':2, 'TA':3, 'Fa':4, 'Po':5},
        'GarageType': {'2Types':1, 'Attchd':2, 'Basment':3, 'BuiltIn':4, 'CarPort':5, 'Detchd':6},
        'GarageFinish': {'Fin':1, 'RFn':2, 'Unf':3},
        'GarageQual': {'Ex':1, 'Gd':2, 'TA':3, 'Fa':4, 'Po':5},
        'GarageCond': {'Ex':1, 'Gd':2, 'TA':3, 'Fa':4, 'Po':5},
        'PavedDrive': {'Y':0, 'P':1, 'N':2},
        'MiscFeature': {'Elev':1, 'Gar2':2, 'Othr':3, 'Shed':4, 'TenC':5},
        'SaleType': {'WD':0, 'CWD':1, 'VWD':2, 'New':3, 'COD':4, 'Con':5, 'ConLw':6, 'ConLI':7, 'ConLD':8, 'Oth':9},
        'SaleCondition': {'Normal':0, 'Abnorml':1, 'AdjLand':2, 'Alloca':3, 'Family':4, 'Partial':5}
    }

    # Apply the dictionary mapping across the entire dataframe at once
    data = data.replace(encoding_maps)

    # 3. Fill specific ordinal features with 0
    zero_fill_cols = ['Alley', 'FireplaceQu', 'GarageType', 'GarageFinish', 
                      'GarageQual', 'GarageCond', 'MiscFeature']
    
    for col in zero_fill_cols:
        if col in data.columns:
            data[col] = data[col].fillna(0)

    # 4. Handle Medians Safely (Prevent Data Leakage)
    # Isolate numeric columns
    numeric_cols = data.select_dtypes(include=['number']).columns

    if is_train:
        # Calculate medians on the train set and save them
        train_medians = data[numeric_cols].median()
        data[numeric_cols] = data[numeric_cols].fillna(train_medians)
        return data, train_medians
    else:
        # Apply the saved train medians to the test set
        if train_medians is None:
            raise ValueError("You must provide train_medians when processing test data.")
        
        # Only fill columns in the test set that exist in the saved medians
        cols_to_fill = numeric_cols.intersection(train_medians.index)
        data[cols_to_fill] = data[cols_to_fill].fillna(train_medians[cols_to_fill])
        return data

In [23]:
# using the function above to clean train_data
train_df, medians = clean_df(train_df)

# using the clean_df function to clean the test data
test_df = clean_df(test_df, is_train = False, train_medians = medians)

/tmp/ipykernel_933/2674939463.py:56: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace(encoding_maps)
/tmp/ipykernel_933/2674939463.py:56: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace(encoding_maps)


## Phase 4 : model training 

In [24]:
# firstly we separate the cleaned data to X(inputs) and y (target)
y = train_df["SalePrice"]
X = train_df.drop(["SalePrice"], axis = 1)

# then we split the data into train data 80% and training data 20%
X_train,X_test,y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state = 43)

# checking the shape 
print(f"The shape of train set is X_train: {X_train.shape} and y_train {y_train.shape}\n")
print(f"The shape of train set is X_test: {X_test.shape} and y_test {y_test.shape}")

The shape of train set is X_train: (1168, 70) and y_train (1168,)

The shape of train set is X_test: (292, 70) and y_test (292,)


In [43]:
#i want to use gridSearch to check this one out 
gd = GradientBoostingRegressor(learning_rate=0.05, max_depth=5, n_estimators=200)
gd.fit(X_train,y_train)

GradientBoostingRegressor(learning_rate=0.05, max_depth=5, n_estimators=200)

In [44]:
# then we use the fitted model to predict if a passenger suvived or not 
gradentb_pred = gd.predict(X_test)

In [45]:
# checking the score 
rmse = root_mean_squared_log_error(y_test, gradentb_pred)

print(f"The root mean squared error of the model : {rmse}")

The root mean squared error of the model :0.14943057671147972


In [53]:
# this is not part of the notebook so it will be hidden
# fitting the model on the full training data 
gd.fit(X, y)

# predicting on the full test data 
y_pred_gbr = gd.predict(test_df)

# creating the submission file
submission_sample = submision_df.copy()
# adding our prediction
submission_sample["SalePrice"] = y_pred_gbr
# converting to csv 
submission_sample.to_csv("submission.csv", index = False)